In [9]:
from pathlib import Path

# ── PARÁMETROS DEL EXPERIMENTO ───────────────────────────────────────────────
OBJECT_ID     = "1a9c1cbf1ca9ca24274623f5a5d0bcdc"
SYMMETRY_TYPE = "axis_sym"

# Rutas de Datos (Renders)
RENDERS_ROOT  = Path("../../data/renders")
IMAGE_SIZE    = 224
ILLUMINATION  = "flat"

# Rutas de Activos 3D (Nuevas)
MESH_PATH = Path(f"../data/objects/curated_{SYMMETRY_TYPE}_obj/{OBJECT_ID}.obj")
SYM_PATH  = Path(f"../data/objects/curated_{SYMMETRY_TYPE}_obj/{OBJECT_ID}.txt")

# Parámetros de Inferencia
EXPERIMENTS = [1, 6, 14, 26]
MODEL_ID    = "allenai/Molmo2-8B"
OUTPUT_DIR  = Path("outputs") / OBJECT_ID / "multiview_experiment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

render_dir = RENDERS_ROOT / SYMMETRY_TYPE / OBJECT_ID / str(IMAGE_SIZE) / ILLUMINATION

print(f"Configuración lista para objeto: {OBJECT_ID}")

Configuración lista para objeto: 1a9c1cbf1ca9ca24274623f5a5d0bcdc


In [10]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def visualize_3d_symmetry(mesh_path, sym_path, size=0.3):
    if not mesh_path.exists():
        print(f"❌ No se encontró el mesh en {mesh_path}")
        return

    # 1. Cargar Mesh
    mesh = o3d.io.read_triangle_mesh(str(mesh_path))
    mesh.compute_vertex_normals()
    mesh.paint_uniform_color([0.8, 0.8, 0.8])

    geometries = [mesh]

    # 2. Cargar Simetrías
    if sym_path.exists():
        with open(sym_path, 'r') as f:
            lines = f.readlines()
            num_planes = int(lines[0].strip())
            for i in range(1, num_planes + 1):
                parts = lines[i].strip().split()
                if parts[0].lower() == 'plane':
                    normal = np.array([float(parts[1]), float(parts[2]), float(parts[3])])
                    point  = np.array([float(parts[4]), float(parts[5]), float(parts[6])])
                    
                    # Crear geometría de plano (basado en tu código de referencia)
                    # Usaremos una línea larga para representar el EJE en lugar de un plano cuadrado
                    # ya que para 'axis_sym' nos interesa el eje central.
                    axis_line = o3d.geometry.LineSet()
                    axis_points = [point - normal * size * 2, point + normal * size * 2]
                    axis_line.points = o3d.utility.Vector3dVector(axis_points)
                    axis_line.lines = o3d.utility.Vector2iVector([[0, 1]])
                    axis_line.paint_uniform_color([1, 0, 0]) # Rojo para el eje
                    geometries.append(axis_line)

    # 3. Renderizar "Offscreen" para mostrar en el notebook
    vis = o3d.visualization.Visualizer()
    vis.create_window(visible=False) 
    for g in geometries:
        vis.add_geometry(g)
    
    vis.get_render_option().background_color = np.asarray([0.95, 0.95, 0.95])
    vis.poll_events()
    vis.update_renderer()
    
    # Capturar imagen
    image = vis.capture_screen_float_buffer(do_render=True)
    vis.destroy_window()
    
    plt.figure(figsize=(8, 8))
    plt.imshow(np.asarray(image))
    plt.title(f"Ground Truth Symmetry: {OBJECT_ID}")
    plt.axis('off')
    plt.show()

visualize_3d_symmetry(MESH_PATH, SYM_PATH)

[Open3D WARNING] GLFW Error: Failed to detect any supported platform
[Open3D WARNING] GLFW initialized for headless rendering.
[Open3D WARNING] GLFW Error: OSMesa: Library not found
[Open3D WARNING] Failed to create window


AttributeError: 'NoneType' object has no attribute 'background_color'